# Country Maps
Generates a styled JPG map for each SIDS country showing the country outline and tile grid.

In [29]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

%matplotlib inline

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar

import contextily as ctx

from ldn.grids import get_gadm
from ldn.utils import ALL_COUNTRIES

from dep_tools.grids import (
    COUNTRIES_AND_CODES as DEP_COUNTRIES_AND_CODES,
)
from ldn.utils import NON_DEP_COUNTRIES

from matplotlib.ticker import FuncFormatter
import pyproj

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
COUNTRIES_PATH = Path("../ldn/gadm_sids.gpkg")
GRID_PATH      = Path("../ldn/sids_all_tiles.geojson")

OUTPUT_DIR = Path("maps")
for subdir in ("", "countries", "countries/Pacific", "countries/Non-Pacific", "regions"):
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

# I already ran this once so no need to rerun.
# get_gadm(countries=ALL_COUNTRIES, overwrite=True) # Update to all countries.

# Load in WGS84, reproject per-map instead of globally
countries_src = gpd.read_file(COUNTRIES_PATH)
grid_src      = gpd.read_file(GRID_PATH)

NON_PACIFIC_CRS = "EPSG:3857"  # Web Mercator
PACIFIC_CRS  = "EPSG:3832"  # WGS 84 / PDC Mercator — centred at 150 °E, handles antimeridian

crs_map = {
    "Pacific": PACIFIC_CRS,
    "Non-Pacific": NON_PACIFIC_CRS,
}

print(f"Loaded {len(countries_src)} countries")
print(f"Loaded {len(grid_src)} grid tiles")

NAME_COL = "COUNTRY"
print(countries_src[NAME_COL].unique())

# Reproject once per CRS — not once per country
countries_3857 = countries_src.to_crs(NON_PACIFIC_CRS)
countries_3832 = countries_src.to_crs(PACIFIC_CRS)
grid_3857      = grid_src.to_crs(NON_PACIFIC_CRS)
grid_3832      = grid_src.to_crs(PACIFIC_CRS)

crs_layers = {
    NON_PACIFIC_CRS: (countries_3857, grid_3857),
    PACIFIC_CRS:  (countries_3832, grid_3832),
}

out_dpi = 75 # Fine for web.

Loaded 60 countries
Loaded 839 grid tiles
['Anguilla' 'Antigua and Barbuda' 'Aruba' 'Bahamas' 'Barbados' 'Belize'
 'Bermuda' 'British Virgin Islands' 'Cayman Islands' 'Cuba' 'Curaçao'
 'Dominica' 'Dominican Republic' 'Grenada' 'Guadeloupe' 'Guyana' 'Haiti'
 'Jamaica' 'Martinique' 'Montserrat' 'Puerto Rico' 'Saint Kitts and Nevis'
 'Saint Lucia' 'Saint Vincent and the Grenadines' 'Sint Maarten'
 'Suriname' 'Trinidad and Tobago' 'Turks and Caicos Islands'
 'Virgin Islands, U.S.' 'American Samoa' 'Cook Islands' 'Fiji'
 'French Polynesia' 'Guam' 'Kiribati' 'Marshall Islands' 'Micronesia'
 'Nauru' 'New Caledonia' 'Niue' 'Northern Mariana Islands' 'Palau'
 'Papua New Guinea' 'Samoa' 'Solomon Islands' 'Timor-Leste' 'Tonga'
 'Tuvalu' 'Vanuatu' 'Cabo Verde' 'Comoros' 'Guinea-Bissau' 'Maldives'
 'Mauritius' 'São Tomé and Príncipe' 'Seychelles' 'Singapore'
 'Pitcairn Islands' 'Tokelau' 'Wallis and Futuna']


In [31]:
DEP_COUNTRY_NAMES = set(DEP_COUNTRIES_AND_CODES.keys())
NON_DEP_COUNTRY_NAMES = set(NON_DEP_COUNTRIES.keys())

def get_region_label(country_name: str) -> str:
    if country_name in DEP_COUNTRY_NAMES:
        return "Pacific"
    return "Non-Pacific"


REGION_COLORS = {
    "Pacific": {
        "boundary": "#FF003C",
        "grid":     "#00CFFF",
    },
    "Non-Pacific": {
        "boundary": "#FF6B00",
        "grid":     "#39FF14",
    },
}

colors = {
    "grey": "#e1e1e1",
    "dark_grey": "#888888",
    "black": "#000000",
    "white": "#FFFFFF",
}

MARGIN = 1.0
PADDING = 0.7


def get_basemap_zoom(title):
    problems = {
        "Pacific": 4,
        "Fiji": 8,
        "Kiribati": 5,
        "Tuvalu": 8,
    }
    return problems.get(title, "auto") # Most are fine with "auto", but some Pacific antimeridian-crossing cases need to avoid very low-res basemap.


def add_north_arrow(ax):
    """Simple north arrow positioned with consistent margin from map edge."""
    size=0.055
    x = 0.97
    y = 0.9
    ax.annotate(
        "",
        xy=(x, y + size), xytext=(x, y),
        xycoords='axes fraction', textcoords='axes fraction',
        arrowprops=dict(arrowstyle="->", color=colors['grey'], lw=2.5),
        zorder=5,
    )
    ax.text(
        x, y + size + 0.005, "N",
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=11, fontweight="bold", color=colors['white'],
        path_effects=[pe.withStroke(linewidth=2, foreground=colors['black'])],
        zorder=5,
    )

# Shared core
def make_map(
    title: str, # e.g. "Country name" or "Region name"
    subtitle: str, # Region name or "X countries / territories"
    geom: gpd.GeoDataFrame,
    grid: gpd.GeoDataFrame,
    output_path: Path,
    color_dict: dict,
    boundary_label: str,
    crs: str,
):
    """Generate and save a styled map.  Shared by country and region modes."""

    color_boundary = color_dict["boundary"]
    color_grid = color_dict["grid"]

    local_tf = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

    bounds = geom.total_bounds
    x_range = bounds[2] - bounds[0]
    y_range = bounds[3] - bounds[1]

    min_extent = max(x_range, y_range, 50_000)
    pad  = min_extent * 0.1
    half = (min_extent + 2 * pad) / 2
    cx   = (bounds[0] + bounds[2]) / 2
    cy   = (bounds[1] + bounds[3]) / 2

    xlim = (cx - half, cx + half)
    ylim = (cy - half, cy + half)

    fig, ax = plt.subplots(figsize=(10, 10), dpi=out_dpi)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


    basemap_zoom = get_basemap_zoom(title)
    print(f"  Country/Region: {title}, Basemap zoom: {basemap_zoom}")

    # Basemap
    ctx.add_basemap(
        ax,
        crs=crs,
        source=ctx.providers.Esri.WorldImagery,
        zoom=basemap_zoom, # Needed for antimeridian-crossing maps. Otherwise basemap is low res.
        attribution=False,
    )
    

    # Grid tiles
    if not grid.empty:
        grid.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color_grid,
            linewidth=1.8,
            alpha=1,
        )

    # Geometry outline
    geom.plot(
        ax=ax,
        facecolor="none",
        edgecolor=color_boundary,
        linewidth=2.5,
    )

    # Single-line title: "Country - Region" where Region is coloured
    ax.text(
        0.5, 1.02,
        f"{title} - ",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=18, fontweight="bold",
        color=colors["black"],
    )
    ax.text(
        0.5, 1.02,
        subtitle,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=18, fontweight="bold",
        color=color_grid,
        path_effects=[pe.withStroke(linewidth=2, foreground=colors["black"])],
    )

    # Scale bar
    scalebar = ScaleBar(
        1, units="m", dimension="si-length",
        location="lower left",
        pad=PADDING, border_pad=MARGIN, sep=3,
        color=colors["white"],
        box_color=colors["black"], box_alpha=0.7,
        font_properties={"size": 11},
    )
    ax.add_artist(scalebar)

    add_north_arrow(ax)

    # Legend
    legend_elements = [
        Line2D([0], [0], color=color_boundary, linewidth=2.5, label=boundary_label),
        Line2D([0], [0], color=color_grid,     linewidth=1.8, label="Analysis grid tiles"),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper left",
        framealpha=0.7, fontsize=11,
        labelcolor=colors["white"],
        facecolor=colors["black"],
        borderpad=PADDING, borderaxespad=MARGIN,
        handlelength=1.5,
    )

    # Tick formatters
    def format_lon(x, _):
        lon, _ = local_tf.transform(x, 0)
        return f"{lon:.2f}°E" if lon >= 0 else f"{abs(lon):.2f}°W"

    def format_lat(y, _):
        _, lat = local_tf.transform(0, y)
        return f"{lat:.2f}°N" if lat >= 0 else f"{abs(lat):.2f}°S"

    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    ax.tick_params(labelsize=10, color=colors["black"])
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_edgecolor(colors["black"])

    ax.text(
        1 - 0.01, 0.01,
        "Basemap © Esri World Imagery | Boundaries © GADM",
        transform=ax.transAxes,
        fontsize=7, color=colors["grey"], va="bottom", ha="right",
    )

    plt.tight_layout()
    fig.savefig(output_path, dpi=out_dpi, format="jpg", bbox_inches="tight", pil_kwargs={"quality": 92})
    plt.close(fig)
    print(f"  Saved to {output_path}")


# Convenience wrappers
def make_country_map(country_name, output_path):
    region_name = get_region_label(country_name)
    crs = crs_map[region_name]
    countries_proj, grid_proj = crs_layers[crs]
    geom       = countries_proj[countries_proj[NAME_COL] == country_name]
    grid_tiles = grid_proj[grid_proj.intersects(geom.union_all())]
    make_map(country_name, region_name, geom, grid_tiles,
             output_path, REGION_COLORS[region_name], "Country boundary", crs)

def make_region_map(region_name, country_set, output_path):
    crs = crs_map[region_name]
    countries_proj, grid_proj = crs_layers[crs]
    geom       = countries_proj[countries_proj[NAME_COL].isin(country_set)]
    grid_tiles = grid_proj[grid_proj.intersects(geom.union_all())]
    make_map(region_name, f"{geom.shape[0]} countries / territories",
             geom, grid_tiles, output_path,
             REGION_COLORS[region_name], "Region boundary", crs)

In [32]:
# Generate country maps
country_names = sorted(countries_src[NAME_COL].unique())
print(f"Generating maps for {len(country_names)} countries...")

for name in country_names:
    print(f"  Processing: {name}")
    subset       = countries_src[countries_src[NAME_COL] == name]
    country_grid = grid_src[grid_src.intersects(subset.union_all())]
    safe_name    = name.replace(" ", "_").replace("/", "-")
    out_path     = OUTPUT_DIR / "countries" / get_region_label(name) / f"{safe_name}.jpg"
    try:
        make_country_map(name, out_path)
    except Exception as e:
        print(f"  Failed for {name}: {e}")

# Generate region maps
REGION_FILTER = {
    "Pacific":     DEP_COUNTRY_NAMES,
    "Non-Pacific": NON_DEP_COUNTRY_NAMES,
}

print("\nGenerating region maps...")
for region_name, country_set in REGION_FILTER.items():
    print(f"  Processing: {region_name}")
    region_geom = countries_src[countries_src[NAME_COL].isin(country_set)]
    region_grid = grid_src[grid_src.intersects(region_geom.union_all())]
    out_path    = OUTPUT_DIR / "regions" / f"{region_name}.jpg"
    try:
        make_region_map(region_name, country_set, out_path)
    except Exception as e:
        print(f"  Failed for {region_name}: {e}")

print(f"\nDone. Region maps saved to '{OUTPUT_DIR / 'regions'}/'")

# This takes a while for Fiji, Kiribati, Tuvalu, and the Pacific region map because they cross the antimeridian.

Generating maps for 60 countries...
  Processing: American Samoa
  Country/Region: American Samoa, Basemap zoom: auto
  Saved to maps/countries/Pacific/American_Samoa.jpg
  Processing: Anguilla
  Country/Region: Anguilla, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Anguilla.jpg
  Processing: Antigua and Barbuda
  Country/Region: Antigua and Barbuda, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Antigua_and_Barbuda.jpg
  Processing: Aruba
  Country/Region: Aruba, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Aruba.jpg
  Processing: Bahamas
  Country/Region: Bahamas, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Bahamas.jpg
  Processing: Barbados
  Country/Region: Barbados, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Barbados.jpg
  Processing: Belize
  Country/Region: Belize, Basemap zoom: auto
  Saved to maps/countries/Non-Pacific/Belize.jpg
  Processing: Bermuda
  Country/Region: Bermuda, Basemap zoom: auto
  Saved to maps/coun